## 文本分类--中文情感分析：微调模型

In [1]:
import torch
from datasets import load_from_disk

### 1. 加载数据集

In [2]:
ds_file_path = "../../../data/datasets/ChnSentiCorp/ds"

# 从磁盘中加载数据集
ds = load_from_disk(ds_file_path)

# 设置数据集数据的格式为torch
ds.set_format("torch")

In [3]:
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_name', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 9600
    })
    validation: Dataset({
        features: ['text', 'label', 'label_name', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1200
    })
    test: Dataset({
        features: ['text', 'label', 'label_name', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1200
    })
})

### 2. 微调预训练模型

#### 2.1 加载预训练模型

In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [5]:
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
device

'mps'

In [6]:
# 加载预训练模型
num_labels = 2
model_name = "bert-base-chinese"

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels).to(device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-chinese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
# 分词器
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [8]:
# 使用模型进行个试算
inputs = tokenizer(ds["train"]["text"][0], return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

outputs = model(**inputs)
outputs

SequenceClassifierOutput(loss=None, logits=tensor([[-0.1968, -0.4771]], device='mps:0', grad_fn=<LinearBackward0>), hidden_states=None, attentions=None)

In [9]:
outputs.loss, outputs["logits"].shape

(None, torch.Size([1, 2]))

In [10]:
outputs.keys()

odict_keys(['logits'])

In [11]:
tokenizer.model_input_names

['input_ids', 'token_type_ids', 'attention_mask']

#### 2.2 定义性能指标

In [12]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metircs(eval_pred):
    # pred是transformers.trainer_utils.EvalPrediction得对象
    labels = eval_pred.label_ids
    preds = eval_pred.predictions.argmax(-1)  # 得到最大值的索引
    # print(type(eval_pred), labels.shape, preds.shape)
    # print(labels[:5], preds[:5])   # [0 0 2 3 1] [0 0 2 3 1]
    
    f1 = f1_score(labels, preds, average="weighted")
    accuracy = accuracy_score(labels, preds)
    print(f"accuracy:{accuracy}, f1:{f1}\n")
    
    return {"accuracy": accuracy, "f1": f1}

#### 2.3 定义训练运行的所有超参数

In [13]:
from transformers import Trainer, TrainingArguments

In [14]:
batch_size = 64
logging_steps = len(ds["train"])

num_train_epochs = 2   # 训练周期数，中文训练太慢了，减少次数

model_output_dir = "../../../models/train/ChnSentiCorp"

trainning_args = TrainingArguments(
    output_dir=model_output_dir,            # 临时数据保存的路径
    num_train_epochs=num_train_epochs,      # 训练周期数，训练几轮
    learning_rate=1e-5,                     # 定义学习率
    per_device_train_batch_size=batch_size, # 定义训练的批次大小
    per_device_eval_batch_size=batch_size,  # 定义验证/测试的批次大小
    weight_decay=0.01,                      # 假如参数权重衰减，防止过拟合
    eval_strategy="epoch",                  # 定义测试执行的策略，可以取值为：no、epoch、steps
    disable_tqdm=False,                     # 是不是禁止tqdm进度条
    logging_steps=logging_steps,            # 记录日志的频率，每隔多少步记录一次日志
    push_to_hub=False,                      # 是否push模型到hub，需要先登录huggingface
    log_level="error"                       # 日志级别：debug,info,warning,error,critical,passive. 默认是warning
)

#### 2.4 训练

In [15]:
trainer = Trainer(
    model=model, 
    args=trainning_args,
    compute_metrics=compute_metircs,
    train_dataset=ds["test"],
    eval_dataset=ds["validation"],
    tokenizer=tokenizer
)

训练会比较耗时。中文情感分析也特耗时了，把训练集缩小一点来训练模型。

In [16]:
%time trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.483161,0.854167,0.853830
2,No log,0.336968,0.893333,0.893331
3,No log,0.310621,0.896667,0.896658


accuracy:0.8541666666666666, f1:0.8538303386881444

accuracy:0.8933333333333333, f1:0.8933309625414889

accuracy:0.8966666666666666, f1:0.8966580531629156

CPU times: user 14.3 s, sys: 12min 53s, total: 13min 7s
Wall time: 22min 29s


TrainOutput(global_step=57, training_loss=0.43143402902703537, metrics={'train_runtime': 1349.8501, 'train_samples_per_second': 2.667, 'train_steps_per_second': 0.042, 'total_flos': 947199799296000.0, 'train_loss': 0.43143402902703537, 'epoch': 3.0})

我用了1200条训练集来训练，花费了22分钟左右。经过三轮训练正确率是`89%`。

In [19]:
# 评估模型
trainer.evaluate()

accuracy:0.8966666666666666, f1:0.8966580531629156



{'eval_loss': 0.31062138080596924,
 'eval_accuracy': 0.8966666666666666,
 'eval_f1': 0.8966580531629156,
 'eval_runtime': 22.6506,
 'eval_samples_per_second': 52.979,
 'eval_steps_per_second': 0.839,
 'epoch': 3.0}

### 3. 保存模型

In [18]:
model_save_dir = f"{model_output_dir}/model-classfication"
model.save_pretrained(model_save_dir)
tokenizer.save_pretrained(model_save_dir)

('../../../models/train/ChnSentiCorp/model-classfication/tokenizer_config.json',
 '../../../models/train/ChnSentiCorp/model-classfication/special_tokens_map.json',
 '../../../models/train/ChnSentiCorp/model-classfication/vocab.txt',
 '../../../models/train/ChnSentiCorp/model-classfication/added_tokens.json',
 '../../../models/train/ChnSentiCorp/model-classfication/tokenizer.json')